***Title: Generative AI Assignment***

***Name: Gedion Sang***

***ID: CS-DA03-26010***

***Date: 08/04/2026***

## Generative AI

### Introduction
Apply understanding of Generative AI and Retrieval-Augmented Generation (RAG) to build a practical pipeline that retrieves relevant document chunks and generates context-aware answers:
I will demonstrate your ability to:

- Apply generative AI concepts to synthesize answers from retrieved document content.
- Extract key information from a PDF document by splitting it into chunks, embedding it using Sentence-Transformers, and storing it in a FAISS vector database for efficient retrieval.
- Demonstrate how retrieval improves generative question-answering by comparing answers generated from document-grounded context versus generic answers.
- Implement a complete RAG pipeline in Python using LangChain, Hugging Face Transformers, and FAISS.
- Practice prompt engineering to structure queries that guide the generative model for clearer and more detailed responses.

### Importing Libraries
This code cell installs and imports all necessary Python libraries for the RAG pipeline. Key libraries include:

*   `langchain` and `langchain-community`: For building and orchestrating the RAG pipeline components.
*   `langchain-huggingface` and `transformers`: For integrating Hugging Face models, especially for embeddings and the Language Model.
*   `sentence-transformers`: Used for generating sentence embeddings.
*   `faiss-cpu`: A library for efficient similarity search and clustering of dense vectors, used here for the vector database.
*   `pypdf`: For loading and parsing PDF documents.
*   `re`: Python's built-in regular expression module for text cleaning.
*   `torch`: The PyTorch deep learning framework, used for handling tensors with the Hugging Face models.
*   `google.colab.files`: For uploading files in Google Colab.

In [1]:
# 1. Install required libraries
!pip install -q langchain langchain-community langchain-huggingface transformers sentence-transformers faiss-cpu pypdf
import re
import re
import torch
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

### Step 1 : Cleaning Function
This function is designed to clean up common issues that arise when extracting text from PDFs, such as spaces being inserted unexpectedly between characters or words
This Python function, `fix_pdf_text`, is designed to clean up common issues found in text extracted from PDFs. It uses regular expressions to:

1.  Remove single spaces between characters that might have been introduced during PDF parsing (e.g., 'C y b e r' becomes 'Cyber').
2.  Remove single spaces at the beginning of a line.
3.  Remove single spaces following a newline character, which can occur when lines are incorrectly wrapped.
4.  Replace multiple whitespace characters with a single space and strip leading/trailing whitespace.

In [2]:
# STEP 1: CLEANING FUNCTION

def fix_pdf_text(text):
    text = re.sub(r'(?<= )([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'^([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'\n([^ ]) ', r'\n\1', text)
    return re.sub(r'\s+', ' ', text).strip()


## Step 2: Upload and Process PDF
---
This code cell handles the uploading and initial processing of a PDF document:

1.  It prompts the user to upload a PDF file using `files.upload()`.
2.  Once uploaded, it gets the path of the uploaded PDF.
3.  A `PyPDFLoader` is used to load the PDF content.
4.  The `fix_pdf_text` function is applied to the content of each page to clean the text.
5.  A `RecursiveCharacterTextSplitter` is initialized to break down the cleaned PDF content into smaller, manageable chunks. This splitting is crucial for effective retrieval, ensuring that each chunk contains sufficient context without being too large.

In [3]:
# STEP 2: UPLOAD AND PROCESS PDF
print('Please upload your PDF file:')
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Load PDF and apply cleaning
loader = PyPDFLoader(pdf_path)
pages = loader.load()

for page in pages:
    page.page_content = fix_pdf_text(page.page_content)

# Split into chunks for retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
    )
chunks = splitter.split_documents(pages)

Please upload your PDF file:


Saving Kompass.pdf to Kompass (1).pdf


## Step 3: Embeddings
This code cell creates embeddings for the document chunks and stores them in a FAISS vector database:

1.  **`HuggingFaceEmbeddings`**: Initializes an embedding model from Hugging Face (`sentence-transformers/all-MiniLM-L6-v2`) to convert text chunks into numerical vector representations.
2.  **`FAISS.from_documents`**: Creates a FAISS vector store by taking the processed `chunks` and their corresponding `embeddings`. FAISS is optimized for fast similarity search on large datasets of vectors.
3.  **`vectorstore.as_retriever`**: Configures the vector store as a retriever, which will be used to fetch the most relevant document chunks based on a query. The `search_kwargs={'k': 5}` specifies that the retriever should return the top 5 most similar documents.

In [4]:
# STEP 3:EMBEDDINGS
print('Creating embeddings and vector store...')
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

Creating embeddings and vector store...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Step 4: Load LLM
This code cell loads the pre-trained FLAN-T5-large model and its tokenizer, which will be used for generating answers:

1.  **`device`**: Automatically detects if a CUDA-enabled GPU is available and sets the device accordingly, otherwise defaults to CPU.
2.  **`model_id = 'google/flan-t5-large'`**: Specifies the identifier for the FLAN-T5-large model.
3.  **`AutoTokenizer.from_pretrained`**: Loads the tokenizer corresponding to the FLAN-T5-large model. The tokenizer converts text into numerical tokens that the model can understand.
4.  **`AutoModelForSeq2SeqLM.from_pretrained(...).to(device)`**: Loads the pre-trained FLAN-T5-large sequence-to-sequence language model and moves it to the determined `device` (GPU or CPU). This model is capable of generating text based on given prompts and context.

In [5]:
 # STEP 4: LOAD LLM (FLAN-T5-LARGE) MANUALLY
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = 'google/flan-t5-large'

print(f'Loading {model_id} on {device}...')

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

Loading google/flan-t5-large on cuda...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## Step 5: RAG Query Function
This Python function, `query_rag`, implements the core logic of the Retrieval-Augmented Generation (RAG) pipeline:

1.  **`retriever.invoke(question)`**: Takes a `question` as input and uses the configured retriever (from Step 3) to find the most `relevant_docs` (document chunks) from the FAISS vector store.
2.  **`context = "\n".join(...)`**: Combines the content of the `relevant_docs` into a single `context` string, separating each document's content with a newline.
3.  **`prompt = f"""Use the following context..."""`**: Constructs a detailed prompt for the Language Model. This prompt is crucial for guiding the model to:
    *   Use the provided `context` to answer the `question`.
    *   State "I don't know" if the answer is not in the context.
    *   Keep the answer concise and relevant.
4.  **`tokenizer(prompt, return_tensors="pt", ...).to(device)`**: Tokenizes the constructed prompt, preparing it as input for the FLAN-T5 model. It ensures the input is truncated if too long and moved to the correct device (GPU/CPU).
5.  **`model.generate(...)`**: Uses the loaded FLAN-T5 model to generate a response based on the tokenized `inputs`. `max_new_tokens` sets the maximum length of the generated response, and `do_sample=False` ensures deterministic generation (no randomness).
6.  **`tokenizer.decode(...)`**: Decodes the model's output tokens back into a human-readable text string, skipping any special tokens used by the model.

In [6]:
# STEP 5: RAG QUERY FUNCTION
def query_rag(question):
    relevant_docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in relevant_docs])

    # Standardized prompt for T5
    input_text = f"answer: {question} context: {context}"
    inputs = tokenizer(input_text,
                       return_tensors="pt",
                       truncation=True,
                       max_length=1024).to(device)

    # Generate response using the model directly
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Step 6: Execution
This code cell demonstrates the execution of the RAG pipeline:

1.  **`print(f'\n--- RAG Analysis Output for {pdf_path} ---')`**: Prints a header to mark the beginning of the RAG output, including the path of the processed PDF.
2.  **`user_query = "Summarize the key information, professional experience, and skills mentioned in this document."`**: Defines the specific question that will be posed to the RAG system.
3.  **`print(query_rag(user_query))`**: Calls the `query_rag` function with the `user_query`. This function orchestrates the retrieval of relevant information and the generation of an answer by the LLM, which is then printed to the console.

In [7]:
# STEP 6: EXECUTION
print(f'\n--- RAG Analysis Output for {pdf_path} ---')
query1 = "Summarize the Required Skills and Qualifications mentioned in this document."
print(query_rag(query1))

# Other Examples
query2 = "Summarize the Key Responsibilities mentioned in this document."
print(query_rag(query2))

query3 = "Summarize what the company offers as mentioned in this document."
print(query_rag(query3))



--- RAG Analysis Output for Kompass (1).pdf ---
The Director of Travel and Tourism is responsible for managing the travel and tourism needs of Strathmore College’s international students. The Director will assist the Director in preparing travel and tourism related documentation, including visa applications, travel documentation, and related administrative requirements. The Director will also assist the Director in preparing meeting agendas, briefing notes, and follow up on action items. The Director will provide support for the personal and professional needs of the Director. The Director will also assist the Director in preparing travel and tourism related documentation, including visa applications, travel documentation, and related administrative requirements. The Director will assist the Director in preparing meeting agendas, briefing notes, and follow up on action items. The Director will assist the Director in preparing travel and tourism related documentation, including visa ap

In [8]:
query4 = "Summarize the Required Skills and Qualifications mentioned in this document."
print("Query 4:\n", query_rag(query4), "\n")

query5 = "Summarize the Key Responsibilities mentioned in this document."
print("Query 5:\n", query_rag(query5), "\n")

query6 = "Summarize what the company offers as mentioned in this document."
print("Query 6:\n", query_rag(query6), "\n")

Query 4:
 The Director of Travel and Tourism is responsible for managing the travel and tourism needs of Strathmore College’s international students. The Director will assist the Director in preparing travel and tourism related documentation, including visa applications, travel documentation, and related administrative requirements. The Director will also assist the Director in preparing meeting agendas, briefing notes, and follow up on action items. The Director will provide support for the personal and professional needs of the Director. The Director will also assist the Director in preparing travel and tourism related documentation, including visa applications, travel documentation, and related administrative requirements. The Director will assist the Director in preparing meeting agendas, briefing notes, and follow up on action items. The Director will assist the Director in preparing travel and tourism related documentation, including visa applications, travel documentation, and r

## Conclusion

This notebook successfully demonstrates the implementation of a Retrieval-Augmented Generation (RAG) pipeline using Python, LangChain, Hugging Face Transformers, and FAISS. We've covered key steps from document ingestion to context-aware answer generation:

1.  **PDF Processing and Cleaning**: A `fix_pdf_text` function was crucial for preprocessing extracted PDF content, ensuring clean text for further steps.
2.  **Document Chunking**: The `RecursiveCharacterTextSplitter` effectively broke down the PDF into manageable chunks, optimizing for retrieval.
3.  **Embedding and Vector Store**: `HuggingFaceEmbeddings` were used to create vector representations of these chunks, which were then stored in a `FAISS` vector database for efficient similarity search.
4.  **LLM Integration**: The `google/flan-t5-large` model was loaded to serve as the generative component of the RAG system.
5.  **RAG Query Function**: The `query_rag` function orchestrated the retrieval of relevant document chunks and the generation of context-grounded answers based on a user query.

This pipeline effectively showcases how combining a retrieval mechanism with a large language model can significantly improve the accuracy and relevance of generated answers, by grounding them in specific, provided documents rather than relying solely on the LLM's pre-trained knowledge. The prompt engineering and adjustment of retrieval parameters (`k`) further enhanced the quality of the generated responses, demonstrating a practical application of Generative AI principles.